In [1]:
from pathlib import Path

MODEL='Version4'

DATASET_PATH=Path('preprocessed_data')

DATASET_PATH.mkdir(parents=True, exist_ok=True)

EVALUATE = True

DATASETS_TO_EXCLUDE = [] #Datasets to exclude from evaluation. If Empty then checks DATASET_TO_CHECK

DATASET_TO_CHECK = ['kkanji2'] #Datasets to show images and example predictions from

In [ ]:
import json
def get_word_files(dataset_path):
    with open(str(dataset_path / 'labels.json'), 'r') as fp:
        data_dict = json.load(fp)
    words_files = list(data_dict.items())
    print(words_files[0])
    print(f"{len(words_files)} words")
    return(words_files)

In [4]:
words_list = []
dataset_list = ['kmnist', 'K49', 'kanjivg', 'kkanji2', 'text_renderer']

for dataset in dataset_list:
    if dataset in DATASET_TO_CHECK:
        words_list.append((get_word_files(DATASET_PATH / dataset), dataset))

('preprocessed_data/kkanji2/0038a0f9c5d87bbe.png.png', '一')
1483999 words


In [ ]:
from typing import Tuple
import tqdm
import torch
from dtrocr.config import DTrOCRConfig
from torch.utils.data import DataLoader
import torch
torch.set_float32_matmul_precision('high')
from dtrocr.model import DTrOCRLMHeadModel


model = DTrOCRLMHeadModel(DTrOCRConfig())
model = torch.compile(model)
model.load_state_dict(torch.load(f'../models/{MODEL}.pt'))
model.to(device=0)



/home/kane/miniconda3/envs/DTrOCR/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OptimizedModule(
  (_orig_mod): DTrOCRLMHeadModel(
    (transformer): DTrOCRModel(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(4, 8), stride=(4, 8))
      )
      (token_embedding): Embedding(32000, 768)
      (positional_embedding): Embedding(256, 768)
      (hidden_layers): ModuleList(
        (0-11): 12 x GPT2Block(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2SdpaAttention(
            (c_attn): Conv1D()
            (c_proj): Conv1D()
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D()
            (c_proj): Conv1D()
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (dropout): Dropout(p=0.1, inplace=False)


In [6]:
from dtrocr.processor import DTrOCRProcessor
from dtrocr.config import DTrOCRConfig
from pathlib import Path
from dataclasses import dataclass
from PIL import Image
from torch.utils.data import Dataset
  
@dataclass
class Word:
    file_path: Path
    transcription: str
    

def get_word(word_file):
    words = []
    words.append(
        Word(
            file_path=word_file[0],
            transcription=word_file[1]
        )
    )
    return words

    
class IAMDataset(Dataset):
    def __init__(self, words: list[Word], config: DTrOCRConfig):
        super(IAMDataset, self).__init__()
        self.words = words
        self.processor = DTrOCRProcessor(config, add_eos_token=True, add_bos_token=True)
        
    def __len__(self):
        return len(self.words)
    
    def __getitem__(self, item):
        inputs = self.processor(
            images=Image.open(self.words[item].file_path).convert('RGB'),
            texts=self.words[item].transcription,
            padding='max_length',
            return_tensors="pt",
            return_labels=True,
        )
        return {
            'pixel_values': inputs.pixel_values[0],
            'input_ids': inputs.input_ids[0],
            'input_attention_mask': inputs.input_attention_mask[0],
            'label_attention_mask': inputs.label_attention_mask[0],
            'labels': inputs.labels[0]
        }

config = DTrOCRConfig(
    # attn_implementation='flash_attention_2'
)




In [ ]:
import tqdm
import multiprocessing as mp
from util.model import evaluate_model
from util.data_processing import get_words_list, WORDSDataset

train_word_records = {}
for pair in words_list:
    if pair[1] in DATASETS_TO_EXCLUDE : continue
    words = get_words_list[pair[0]]
    train_word_records[pair[1]] = words
    data = WORDSDataset(words=words, config=config)
    dataloader = DataLoader(data, batch_size=32, shuffle=True, num_workers=mp.cpu_count())
    loss, accuracy = evaluate_model(model, dataloader)
            
    print(f"{pair[1]} loss: {loss}, accuracy: {accuracy}")

Building dataset: 100%|██████████| 1483999/1483999 [04:25<00:00, 5595.70it/s] 


Word(file_path='preprocessed_data/kkanji2/0038a0f9c5d87bbe.png.png', transcription='一')


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/home/kane/miniconda3/envs/DTrOCR/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Evaluating test set: 100%|██████████| 46375/46375 [3:15:46<00:00,  3.95it/s]  

kkanji2 loss: 4.984883695926307, accuracy: 0.4845399147461688


In [8]:
from dtrocr.processor import DTrOCRProcessor
from dtrocr.config import DTrOCRConfig

model.eval()
model.to('cpu')
test_processor = DTrOCRProcessor(DTrOCRConfig())

/home/kane/miniconda3/envs/DTrOCR/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
import matplotlib

matplotlib.rc('font', family='TakaoPGothic')

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from util.model import test
from PIL import Image

print(train_word_records[DATASET_TO_CHECK[0]])

for ds in DATASET_TO_CHECK:
    for test_word_record in train_word_records[ds[0]][:10]:
        test(model, test_word_record, 10)

In [ ]:
tokeniser = test_processor.tokeniser
print(tokeniser.pad_token)
print(tokeniser.eos_token)
print(tokeniser.bos_token)
print(tokeniser.model_max_length)
print(tokeniser.sep_token_id)

print(tokeniser('<s>[SEP]'))